## Exercise 5: Variance reduction method

In [1]:
import numpy as np
from scipy import stats

Part 1 — Crude Montecarlo

In [2]:
np.random.seed(42)
n = 100

U = np.random.uniform(0, 1, n)
X = np.exp(U)

theta_mc = np.mean(X)
se = np.std(X, ddof=1) / np.sqrt(n)
ci = stats.t.interval(0.95, df=n-1, loc=theta_mc, scale=se)

print(f"True value:     {np.e - 1:.6f}")
print(f"MC estimate:    {theta_mc:.6f}")
print(f"95% CI:         ({ci[0]:.6f}, {ci[1]:.6f})")
print(f"Variance of Xi: {np.var(X, ddof=1):.6f}  (theory: 0.2420)")

True value:     1.718282
MC estimate:    1.672061
95% CI:         (1.572824, 1.771299)
Variance of Xi: 0.250134  (theory: 0.2420)


part 2 — antithetic vaariables

In [3]:
np.random.seed(42)
n = 100

U = np.random.uniform(0, 1, n)
Y = (np.exp(U) + np.exp(1 - U)) / 2

theta_av = np.mean(Y)
se_av = np.std(Y, ddof=1) / np.sqrt(n)
ci_av = stats.t.interval(0.95, df=n-1, loc=theta_av, scale=se_av)

print(f"True value:     {np.e - 1:.6f}")
print(f"AV estimate:    {theta_av:.6f}")
print(f"95% CI:         ({ci_av[0]:.6f}, {ci_av[1]:.6f})")
print(f"Variance of Yi: {np.var(Y, ddof=1):.6f}  (theory: 0.0039)")
print(f"Variance reduction: {np.var(X, ddof=1)/np.var(Y, ddof=1):.1f}x")

True value:     1.718282
AV estimate:    1.722602
95% CI:         (1.710188, 1.735016)
Variance of Yi: 0.003914  (theory: 0.0039)
Variance reduction: 63.9x


Part 3 — Control Variates

In [4]:
np.random.seed(42)
n = 100

U = np.random.uniform(0, 1, n)
X = np.exp(U)

# Optimal c = -Cov(e^U, U) / Var(U)
# Estimated from the same sample
cov_xu = np.cov(X, U, ddof=1)[0, 1]
var_u  = np.var(U, ddof=1)
c = -cov_xu / var_u

Y = X + c * (U - 0.5)  # E[U] = 0.5

theta_cv = np.mean(Y)
se_cv = np.std(Y, ddof=1) / np.sqrt(n)
ci_cv = stats.t.interval(0.95, df=n-1, loc=theta_cv, scale=se_cv)

print(f"True value:     {np.e - 1:.6f}")
print(f"CV estimate:    {theta_cv:.6f}")
print(f"95% CI:         ({ci_cv[0]:.6f}, {ci_cv[1]:.6f})")
print(f"Optimal c:      {c:.4f}  (theory: -1.69)")
print(f"Variance of Yi: {np.var(Y, ddof=1):.6f}  (theory: 0.0039)")
print(f"Variance reduction: {np.var(X, ddof=1)/np.var(Y, ddof=1):.1f}x")

True value:     1.718282
CV estimate:    1.721805
95% CI:         (1.709478, 1.734132)
Optimal c:      -1.6682  (theory: -1.69)
Variance of Yi: 0.003860  (theory: 0.0039)
Variance reduction: 64.8x


Part 4 — Stratified Sampling

In [5]:
np.random.seed(42)
n = 100
k = 10  # strata

# For each of n repetitions, draw 1 sample from each of k strata
Y = np.zeros(n)
for i in range(n):
    for j in range(1, k+1):
        U_ij = np.random.uniform((j-1)/k, j/k)
        Y[i] += np.exp(U_ij)
    Y[i] /= k

theta_ss = np.mean(Y)
se_ss = np.std(Y, ddof=1) / np.sqrt(n)
ci_ss = stats.t.interval(0.95, df=n-1, loc=theta_ss, scale=se_ss)

print(f"True value:     {np.e - 1:.6f}")
print(f"SS estimate:    {theta_ss:.6f}")
print(f"95% CI:         ({ci_ss[0]:.6f}, {ci_ss[1]:.6f})")
print(f"Variance of Yi: {np.var(Y, ddof=1):.6f}")
print(f"Variance reduction: {np.var(X, ddof=1)/np.var(Y, ddof=1):.1f}x")

True value:     1.718282
SS estimate:    1.716495
95% CI:         (1.713173, 1.719817)
Variance of Yi: 0.000280
Variance reduction: 892.4x


Part 7 — importance sampling for P(Z > a)

In [6]:
np.random.seed(42)
n = 10000

for a in [2, 4]:
    # Crude MC
    Z = np.random.normal(0, 1, n)
    p_mc = np.mean(Z > a)
    
    # Importance sampling: g = N(a, 1)
    Y = np.random.normal(a, 1, n)
    weights = stats.norm.pdf(Y, 0, 1) / stats.norm.pdf(Y, a, 1)
    p_is = np.mean((Y > a) * weights)
    var_is = np.var((Y > a) * weights, ddof=1)
    
    true_p = 1 - stats.norm.cdf(a)
    print(f"--- a = {a} ---")
    print(f"True P(Z>{a}):  {true_p:.6f}")
    print(f"Crude MC:       {p_mc:.6f}  (var: {np.var(Z>a, ddof=1):.6f})")
    print(f"IS estimate:    {p_is:.6f}  (var: {var_is:.6f})")
    print(f"Variance reduction: {np.var(Z>a, ddof=1)/var_is:.1f}x\n")

--- a = 2 ---
True P(Z>2):  0.022750
Crude MC:       0.023700  (var: 0.023141)
IS estimate:    0.023275  (var: 0.001259)
Variance reduction: 18.4x

--- a = 4 ---
True P(Z>4):  0.000032
Crude MC:       0.000000  (var: 0.000000)
IS estimate:    0.000031  (var: 0.000000)
Variance reduction: 0.0x



zoom in on a=4 with more detail

In [7]:
np.random.seed(42)
n = 10000
a = 4

# Crude MC - how many hits?
Z = np.random.normal(0, 1, n)
hits = np.sum(Z > a)

# IS
Y = np.random.normal(a, 1, n)
weights = stats.norm.pdf(Y, 0, 1) / stats.norm.pdf(Y, a, 1)
p_is = np.mean((Y > a) * weights)
se_is = np.std((Y > a) * weights, ddof=1) / np.sqrt(n)
ci_is = stats.t.interval(0.95, df=n-1, loc=p_is, scale=se_is)

true_p = 1 - stats.norm.cdf(a)
print(f"True P(Z>4):     {true_p:.8f}")
print(f"Crude MC hits:   {hits} out of {n} (estimate = 0!)")
print(f"IS estimate:     {p_is:.8f}")
print(f"IS 95% CI:       ({ci_is[0]:.8f}, {ci_is[1]:.8f})")
print(f"CI contains true value: {ci_is[0] < true_p < ci_is[1]}")

True P(Z>4):     0.00003167
Crude MC hits:   0 out of 10000 (estimate = 0!)
IS estimate:     0.00003298
IS 95% CI:       (0.00003162, 0.00003435)
CI contains true value: True


Part 8 — Importance Sampling with g(x) = λe^(-λx)

In [8]:
np.random.seed(42)
n = 10000

# f(x) = 1 on [0,1], h(x) = e^x
# g(x) = lambda * exp(-lambda * x), truncated to [0,1]
# weight = f(x)/g(x) = 1 / (lambda * exp(-lambda * x))

results = []
for lam in [0.5, 1.0, 1.5, 2.0, 3.0]:
    # Sample from truncated exponential on [0,1]
    # CDF: G(x) = (1 - e^(-lam*x)) / (1 - e^(-lam))
    # Inverse CDF: x = -log(1 - u*(1-e^(-lam))) / lam
    U = np.random.uniform(0, 1, n)
    Y = -np.log(1 - U * (1 - np.exp(-lam))) / lam
    
    # g(x) on [0,1] (normalized)
    g_Y = lam * np.exp(-lam * Y) / (1 - np.exp(-lam))
    
    weights = np.exp(Y) / g_Y  # h(x)*f(x)/g(x), f(x)=1
    theta_is = np.mean(weights)
    var_is = np.var(weights, ddof=1)
    
    results.append((lam, theta_is, var_is))
    print(f"λ={lam:.1f}  estimate={theta_is:.6f}  variance={var_is:.6f}")

print(f"\nTrue value: {np.e-1:.6f}")

λ=0.5  estimate=1.702159  variance=0.558342
λ=1.0  estimate=1.735579  variance=1.089478
λ=1.5  estimate=1.710397  variance=1.777069
λ=2.0  estimate=1.712496  variance=2.817768
λ=3.0  estimate=1.708263  variance=6.402247

True value: 1.718282


Part 9 — Optimal IS using g(x) ∝ e^x 

In [9]:
# If g(x) = e^x / (e-1) on [0,1]  (proportional to h(x)*f(x) = e^x)
# then weight = f(x)/g(x) = (e-1) * e^(-x) -- constant variance!
# 
# Sample from g(x): inverse CDF
# G(x) = (e^x - 1)/(e - 1)  =>  x = log(1 + u*(e-1))

np.random.seed(42)
n = 10000

U = np.random.uniform(0, 1, n)
Y = np.log(1 + U * (np.e - 1))

g_Y = np.exp(Y) / (np.e - 1)
weights = np.exp(Y) / g_Y  # = (e-1) for all samples!

theta_opt = np.mean(weights)
var_opt   = np.var(weights, ddof=1)

print(f"True value:       {np.e-1:.6f}")
print(f"Optimal IS est:   {theta_opt:.6f}")
print(f"Variance:         {var_opt:.10f}  (should be ~0!)")
print(f"\nAll weights equal {np.e-1:.6f}: {np.allclose(weights, np.e-1)}")
print(f"\nSummary of all methods:")
print(f"  Crude MC:        var = 0.2501")
print(f"  Antithetic:      var = 0.0039")
print(f"  Control variate: var = 0.0039")
print(f"  Stratified k=10: var = 0.0003")
print(f"  IS exp (λ=0.5):  var = 0.5583")
print(f"  IS optimal g:    var = {var_opt:.6f}")

True value:       1.718282
Optimal IS est:   1.718282
Variance:         0.0000000000  (should be ~0!)

All weights equal 1.718282: True

Summary of all methods:
  Crude MC:        var = 0.2501
  Antithetic:      var = 0.0039
  Control variate: var = 0.0039
  Stratified k=10: var = 0.0003
  IS exp (λ=0.5):  var = 0.5583
  IS optimal g:    var = 0.000000
